# BlackPearl Morphix — Stage 1 (v2): Learning Tissue Layouts with Flow Matching

**What this notebook does:** trains a real generative model on real spatial single-cell data, then measures honestly how close its generated tissue is to held-out real tissue.

1. **Data:** a public spatial dataset where every cell has an (x, y) position and a cell type.
2. **Patches:** the tissue is cut into small neighbourhoods of 64 cells. Scattered blocks of tissue are held out for testing.
3. **Model:** a flow-matching network learns a velocity field $\frac{dx_t}{dt} = v_\theta(x_t, t)$ that moves random noise (t = 0) into realistic tissue patches (t = 1). Each block is attention over a distance-weighted cell graph.
4. **Physics term:** an optional overlap penalty discourages cells from sitting on top of each other (the $\mathcal{L}_{overlap}$ idea from the spec).
5. **Evaluation:** generated patches are compared with held-out real patches, against baselines, with a low-data ablation (physics term on vs off).

**What it does *not* do yet:** there is no genomic conditioning in Stage 1. The model learns *what normal tissue arrangement looks like*; conditioning on perturbations is Stage 2.

Runtime: **GPU** (Runtime → Change runtime type → T4 GPU). Full run with ablation ≈ 20–25 min on a T4.

**Changes from v1:** block-based test split (v1's strip had a very different cell-type mix from training), type-shuffled baseline built from training data (v1 leaked test data into it), held-out position and type losses logged separately, overlap penalty rescaled, and the physics ablation moved to a low-data setting.

In [ ]:
!pip -q install anndata

import os, json, time, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('WARNING: no GPU found. Training will be very slow. Use Runtime -> Change runtime type -> T4 GPU.')
os.makedirs('results', exist_ok=True)

## Configuration
`dataset`: `'seqfish'` = mouse embryo seqFISH (~19k cells, ~22 types, good for a first run); `'imc'` = human breast cancer imaging mass cytometry (~4.7k cells, closer to the cancer theme but small).
`overlap_weight`: strength of the physics penalty in the main model.

**Physics ablation:** in v1 the overlap penalty stayed near zero, because with 20,000 training patches the model learns not to overlap cells from the data alone. So the ablation now asks the more interesting question: **does the physics term help when data is scarce?** It trains two small-data models (with and without the penalty) and compares them.

In [ ]:
CFG = dict(
    dataset='seqfish',
    n_cells=64,             # cells per tissue patch
    test_frac=0.2,          # fraction of tissue held out for testing (as scattered blocks)
    n_blocks=4,             # tissue is cut into n_blocks x n_blocks blocks
    n_train_patches=20000,
    n_test_patches=2000,
    d_model=128, n_layers=4, n_heads=4,
    batch_size=128, steps=15000, lr=3e-4, warmup=500, ema=0.999,
    type_weight=1.0,
    overlap_weight=10.0,
    sample_steps=100,
    n_eval=500,
    seed=0,
)
RUN_ABLATION = True                 # low-data physics ablation (two extra short runs, ~10 min)
ABLATION = dict(n_train_patches=500, steps=6000, overlap_weight=10.0)

rng = np.random.default_rng(CFG['seed'])
torch.manual_seed(CFG['seed'])

## 1. Data loading and tissue patches

In [ ]:
# ---------- DATA CELL (numpy / scipy only) ----------
import os, sys, subprocess, urllib.request
import numpy as np
from scipy.spatial import cKDTree

DATASETS = {
    # Mouse embryo seqFISH (Lohoff et al. 2021), ~19k cells, ~22 cell types
    'seqfish': dict(url='https://ndownloader.figshare.com/files/26098364', file='seqfish.h5ad',
                    key='celltype_mapped_refined', sq='seqfish'),
    # Human breast cancer imaging mass cytometry (Jackson et al. 2020), ~4.7k cells
    'imc': dict(url='https://ndownloader.figshare.com/files/26098406', file='imc.h5ad',
                key='cell type', sq='imc'),
}


def load_anndata(name, data_dir='data'):
    """Download a public spatial dataset as AnnData. Falls back to squidpy if the direct link fails."""
    import anndata as ad
    info = DATASETS[name]
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, info['file'])
    if os.path.exists(path):
        return ad.read_h5ad(path)
    try:
        print(f'Downloading {name} ...')
        urllib.request.urlretrieve(info['url'], path)
        return ad.read_h5ad(path)
    except Exception as e:
        print(f'Direct download failed ({e}). Falling back to squidpy (slower install)...')
        if os.path.exists(path):
            os.remove(path)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'squidpy'], check=True)
        import squidpy as sq
        adata = getattr(sq.datasets, info['sq'])()
        adata.write_h5ad(path)
        return adata


def extract_cells(adata, preferred_key=None):
    """Return 2D coordinates, integer cell types, type names and the obs column used."""
    coords = np.asarray(adata.obsm['spatial'], dtype=np.float64)[:, :2]
    key = preferred_key if preferred_key in adata.obs.columns else None
    if key is None:
        cands = [c for c in adata.obs.columns
                 if (str(adata.obs[c].dtype) in ('category', 'object'))
                 and 3 <= adata.obs[c].nunique() <= 60]
        if not cands:
            raise ValueError(f'No cell-type column found. obs columns: {list(adata.obs.columns)}')
        key = cands[0]
    labels = adata.obs[key].astype(str).values
    type_names, types = np.unique(labels, return_inverse=True)
    return coords, types.astype(np.int64), [str(t) for t in type_names], key


def normalise_spacing(coords):
    """Rescale so the median nearest-neighbour distance between cells is 1 ('cell spacing units')."""
    d, _ = cKDTree(coords).query(coords, k=2)
    nn = d[:, 1]
    d_nn = float(np.median(nn[nn > 0]))
    return coords / d_nn, d_nn


def block_split(coords, test_frac=0.2, n_blocks=4, seed=0):
    """Cut the tissue into an n_blocks x n_blocks grid and hold out random blocks as test tissue.
    Unlike a single strip, test blocks are scattered across the tissue, so train and test
    contain similar anatomical regions and cell-type mixes. Returns a boolean test mask per cell."""
    rng = np.random.default_rng(seed)
    lo, hi = coords.min(0), coords.max(0)
    b = np.minimum(((coords - lo) / (hi - lo + 1e-9) * n_blocks).astype(int), n_blocks - 1)
    block_id = b[:, 0] * n_blocks + b[:, 1]
    ids = rng.permutation(np.unique(block_id))
    counts = np.bincount(block_id, minlength=n_blocks ** 2)
    test_blocks, n_test = [], 0
    for i in ids:
        if n_test >= test_frac * len(coords):
            break
        test_blocks.append(i)
        n_test += counts[i]
    return np.isin(block_id, test_blocks)


def make_patches(coords, types, in_split, n_patches, n_cells, rng):
    """A patch = the n_cells nearest cells to a random centre cell, centred at their centroid.
    Neighbours are searched in the WHOLE tissue and a patch is kept only if every one of its cells
    belongs to this split, so patches are never cut off by a block edge and train/test never share cells."""
    tree = cKDTree(coords)
    pool = np.flatnonzero(in_split)
    _, idx_all = tree.query(coords[pool], k=n_cells)
    clean = in_split[idx_all].all(1)
    if clean.sum() == 0:
        raise ValueError('No clean patches: use fewer/larger blocks or fewer cells per patch.')
    idx = idx_all[clean][rng.integers(0, clean.sum(), n_patches)]
    idx = np.take_along_axis(idx, rng.permuted(np.tile(np.arange(n_cells), (n_patches, 1)), axis=1), axis=1)
    pos = coords[idx]
    pos = pos - pos.mean(axis=1, keepdims=True)
    return pos.astype(np.float32), types[idx].astype(np.int64)

In [ ]:
adata = load_anndata(CFG['dataset'])
coords_raw, types, TYPE_NAMES, type_key = extract_cells(adata, DATASETS[CFG['dataset']]['key'])
K = len(TYPE_NAMES)
coords, d_nn = normalise_spacing(coords_raw)
is_test = block_split(coords, CFG['test_frac'], CFG['n_blocks'], CFG['seed'])
print(f'{len(coords)} cells, {K} cell types (column "{type_key}")')
print(f'median cell spacing = {d_nn:.3f} raw units -> rescaled to 1.0')
print(f'train cells: {(~is_test).sum()} | held-out test cells: {is_test.sum()}')

train_pos, train_typ = make_patches(coords, types, ~is_test, CFG['n_train_patches'], CFG['n_cells'], rng)
test_pos, test_typ = make_patches(coords, types, is_test, CFG['n_test_patches'], CFG['n_cells'], rng)
_c_tr = np.bincount(train_typ.ravel(), minlength=K) / train_typ.size
_c_te = np.bincount(test_typ.ravel(), minlength=K) / test_typ.size
print(f'cell-type mix difference between train and test patches (TVD): {0.5 * np.abs(_c_tr - _c_te).sum():.3f} '
      f'(v1 strip split was 0.565; lower = fairer test)')

POS_SCALE = float(train_pos.std())
_nn = np.concatenate([np.sort(np.linalg.norm(p[:, None] - p[None], axis=-1), 1)[:, 1] for p in train_pos[:500]])
D_MIN = float(np.quantile(_nn, 0.02))    # only 2% of real cells are closer than this to a neighbour
TYPE_FREQ = np.bincount(train_typ.ravel(), minlength=K) / train_typ.size
print(f'position scale = {POS_SCALE:.3f} | overlap threshold D_MIN = {D_MIN:.3f} cell spacings')

PALETTE = [plt.get_cmap('tab20')(i) for i in range(20)] + [plt.get_cmap('tab20b')(i) for i in range(20)]
def colours(t): return [PALETTE[i % len(PALETTE)] for i in t]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(coords[~is_test, 0], coords[~is_test, 1], c=colours(types[~is_test]), s=2)
ax.scatter(coords[is_test, 0], coords[is_test, 1], c=colours(types[is_test]), s=2, alpha=0.25)
ax.set_title(f'{CFG["dataset"]}: cells coloured by type (faded blocks = held-out test tissue)')
ax.set_aspect('equal'); ax.axis('off')
handles = [plt.Line2D([], [], marker='o', ls='', color=PALETTE[i % len(PALETTE)], label=n) for i, n in enumerate(TYPE_NAMES)]
ax.legend(handles=handles, fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
plt.tight_layout(); plt.savefig('results/tissue_overview.png', dpi=150); plt.show()

## 2. Model: flow matching over a set of cells
Each cell is a token $[x, y, \text{one-hot type}]$. Training takes a real patch $x_1$, a noise sample $x_0$, a random time $t$, the point $x_t = (1-t)x_0 + t x_1$, and trains $v_\theta(x_t, t)$ to predict the straight-line velocity $x_1 - x_0$. Sampling integrates from noise to tissue. Cell types are generated jointly as a continuous relaxation and read out with argmax.

**Physics term:** from $x_t$ and the predicted velocity we can estimate the final layout $\hat{x}_1 = x_t + (1-t)v_\theta$ and penalise cell pairs closer than `D_MIN`.

In [ ]:
# ---------- MODEL CELL (PyTorch) ----------
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def pairwise_dist(p):
    """Pairwise distances with a safe gradient at zero (torch.cdist can give NaN gradients there)."""
    diff = p[:, :, None, :] - p[:, None, :, :]
    return torch.sqrt((diff ** 2).sum(-1) + 1e-8)


class TimeEmbedding(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d))

    def forward(self, t):                       # t: [B] in [0, 1]
        half = self.d // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        a = 1000.0 * t[:, None] * freqs[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], dim=-1))


class DistanceBias(nn.Module):
    """Turns pairwise cell distances into per-head attention biases: nearby cells 'talk' more.
    This makes each block a message-passing layer on a fully connected, distance-weighted cell graph."""
    def __init__(self, n_heads, n_rbf=16, max_dist=4.0):
        super().__init__()
        self.register_buffer('centres', torch.linspace(0, max_dist, n_rbf))
        self.gamma = (n_rbf / max_dist) ** 2
        self.proj = nn.Linear(n_rbf, n_heads)

    def forward(self, pos):                     # pos: [B, N, 2]
        d = pairwise_dist(pos)                  # [B, N, N]
        rbf = torch.exp(-self.gamma * (d[..., None] - self.centres) ** 2)
        return self.proj(rbf).permute(0, 3, 1, 2)   # [B, H, N, N]


class Block(nn.Module):
    """Transformer block with time-conditioned adaptive LayerNorm (as in DiT)."""
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.bias = DistanceBias(n_heads)
        self.ada = nn.Linear(d, 6 * d)
        nn.init.zeros_(self.ada.weight)
        nn.init.zeros_(self.ada.bias)

    def forward(self, h, c, pos):
        B, N, _ = h.shape
        s1, b1, g1, s2, b2, g2 = self.ada(c)[:, None].chunk(6, dim=-1)
        mask = self.bias(pos).reshape(B * self.n_heads, N, N)
        x = self.norm1(h) * (1 + s1) + b1
        h = h + g1 * self.attn(x, x, x, attn_mask=mask, need_weights=False)[0]
        x = self.norm2(h) * (1 + s2) + b2
        return h + g2 * self.mlp(x)


class TissueFlow(nn.Module):
    """Velocity field v_theta(x_t, t) over a SET of cells. Each cell's state = [x, y, soft one-hot type].
    No positional encoding over cell index, so the model is permutation-equivariant (cell order is meaningless)."""
    def __init__(self, n_types, d=128, n_layers=4, n_heads=4):
        super().__init__()
        self.dim = 2 + n_types
        self.inp = nn.Linear(self.dim, d)
        self.time = TimeEmbedding(d)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.out_norm = nn.LayerNorm(d)
        self.out = nn.Linear(d, self.dim)

    def forward(self, x, t):                    # x: [B, N, 2+K], t: [B]
        c = self.time(t)
        h = self.inp(x) + c[:, None]
        for blk in self.blocks:
            h = blk(h, c, x[..., :2])
        return self.out(self.out_norm(h))


def random_rotate(pos):
    """Tissue has no preferred orientation: random rotation + reflection as augmentation."""
    B = pos.shape[0]
    th = torch.rand(B, device=pos.device) * 2 * math.pi
    c, s = th.cos(), th.sin()
    R = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)   # [B, 2, 2]
    pos = pos @ R.transpose(1, 2)
    flip = (torch.rand(B, 1, device=pos.device) < 0.5).float() * 2 - 1
    return torch.cat([pos[..., :1] * flip[..., None], pos[..., 1:]], dim=-1)


def flow_matching_loss(model, pos, typ, n_types, pos_scale, d_min, type_weight=1.0, overlap_weight=0.0):
    """Conditional flow matching (straight paths from Gaussian noise x0 to real tissue x1),
    plus an optional physics term: penalise predicted final cells that sit closer than d_min."""
    B, N, _ = pos.shape
    x1 = torch.cat([pos / pos_scale, F.one_hot(typ, n_types).float()], dim=-1)
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=pos.device)
    tt = t[:, None, None]
    xt = (1 - tt) * x0 + tt * x1
    v = model(xt, t)
    err = (v - (x1 - x0)) ** 2
    loss_pos, loss_type = err[..., :2].mean(), err[..., 2:].mean()
    loss = loss_pos + type_weight * loss_type
    loss_ov = torch.zeros((), device=pos.device)
    if overlap_weight > 0:
        x1_hat = (xt[..., :2] + (1 - tt) * v[..., :2]) * pos_scale      # predicted final positions, spacing units
        d = pairwise_dist(x1_hat) + torch.eye(N, device=pos.device) * 1e3
        viol = F.relu(d_min - d) ** 2
        loss_ov = (viol.sum((1, 2)) / N * t).mean()       # energy per cell; trust the estimate more near t=1
        loss = loss + overlap_weight * loss_ov
    return loss, {'pos': loss_pos.item(), 'type': loss_type.item(), 'overlap': loss_ov.item()}


@torch.no_grad()
def sample(model, n, n_cells, n_types, pos_scale, steps=100, batch=250, device='cpu', keep_traj=False):
    """Integrate dx/dt = v_theta(x, t) from noise (t=0) to tissue (t=1) with Euler steps."""
    model.eval()
    all_pos, all_typ, traj = [], [], []
    for i in range(0, n, batch):
        m = min(batch, n - i)
        x = torch.randn(m, n_cells, 2 + n_types, device=device)
        for k in range(steps):
            if keep_traj and i == 0 and k % (steps // 4) == 0:
                traj.append(x[0].clone().cpu())
            t = torch.full((m,), k / steps, device=device)
            x = x + model(x, t) / steps
        if keep_traj and i == 0:
            traj.append(x[0].clone().cpu())
        pos = x[..., :2] * pos_scale
        pos = pos - pos.mean(1, keepdim=True)
        all_pos.append(pos.cpu().numpy())
        all_typ.append(x[..., 2:].argmax(-1).cpu().numpy())
    model.train()
    out = (np.concatenate(all_pos), np.concatenate(all_typ))
    return (*out, traj) if keep_traj else out

## 3. Training

In [ ]:
train_pos_t = torch.tensor(train_pos, device=DEVICE)
train_typ_t = torch.tensor(train_typ, device=DEVICE)
test_pos_t = torch.tensor(test_pos, device=DEVICE)
test_typ_t = torch.tensor(test_typ, device=DEVICE)


def train_model(overlap_weight, tag, n_patches=None, steps=None):
    torch.manual_seed(CFG['seed'])
    n_avail = n_patches or len(train_pos_t)
    model = TissueFlow(K, CFG['d_model'], CFG['n_layers'], CFG['n_heads']).to(DEVICE)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.01)
    total, warm = steps or CFG['steps'], CFG['warmup']
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    print(f'[{tag}] {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')
    log, t0 = [], time.time()
    for step in range(total + 1):
        idx = torch.randint(0, n_avail, (CFG['batch_size'],), device=DEVICE)
        loss, parts = flow_matching_loss(model, random_rotate(train_pos_t[idx]), train_typ_t[idx], K,
                                         POS_SCALE, D_MIN, CFG['type_weight'], overlap_weight)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        with torch.no_grad():
            for pe, p in zip(ema.parameters(), model.parameters()):
                pe.lerp_(p, 1 - CFG['ema'])
        if step % 100 == 0:
            row = {'step': step, 'train': loss.item(), **parts}
            if step % 1000 == 0:
                with torch.no_grad():
                    j = torch.randint(0, len(test_pos_t), (512,), device=DEVICE)
                    torch.manual_seed(1234)    # same noise every evaluation, so the curve isn't jittery
                    vl, vp = flow_matching_loss(ema, test_pos_t[j], test_typ_t[j], K, POS_SCALE, D_MIN, 1.0, 0.0)
                    torch.manual_seed(CFG['seed'] + step)
                row.update(test_flow=vl.item(), test_pos=vp['pos'], test_type=vp['type'])
                print(f'[{tag}] step {step:6d} | train {loss.item():.4f} | held-out: position {vp["pos"]:.4f}, '
                      f'type {vp["type"]:.4f} | overlap {parts["overlap"]:.5f} | {time.time() - t0:.0f}s')
            log.append(row)
    torch.save({'model': ema.state_dict(), 'cfg': CFG, 'types': TYPE_NAMES, 'pos_scale': POS_SCALE,
                'd_min': D_MIN}, f'results/morphix_stage1_{tag}.pt')
    return ema, pd.DataFrame(log)


models, logs = {}, {}
models['main'], logs['main'] = train_model(CFG['overlap_weight'], 'main')
if RUN_ABLATION:
    for tag, w in [('lowdata_physics', ABLATION['overlap_weight']), ('lowdata_no_physics', 0.0)]:
        models[tag], logs[tag] = train_model(w, tag, ABLATION['n_train_patches'], ABLATION['steps'])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for tag, lg in logs.items():
    v = lg.dropna(subset=['test_flow'])
    ax[0].plot(v['step'], v['test_pos'], 'o-', label=tag)
    ax[1].plot(v['step'], v['test_type'], 'o-', label=tag)
    ax[2].plot(lg['step'], lg['overlap'], label=tag)
ax[0].set_title('Held-out loss: cell POSITIONS'); ax[1].set_title('Held-out loss: cell TYPES')
ax[2].set_title('Overlap penalty (training)'); ax[2].set_yscale('symlog', linthresh=1e-5)
for a in ax:
    a.set_xlabel('step'); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig('results/training_curves.png', dpi=150); plt.show()

## 4. Evaluation against held-out tissue
All scores compare a set of patches with **held-out real test patches** (lower = more like real tissue):

| Metric | What it checks |
|---|---|
| NN-distance W1 | Are cells spaced like real cells? (Wasserstein distance between nearest-neighbour distance distributions) |
| Pair-distance W1 | Overall patch geometry |
| Patch-radius W1 | Are patches as compact/spread as real ones? |
| Composition TVD | Right mix of cell types? |
| Neighbourhood TVD | Are the right cell types next to each other? (the spatial organisation) |
| Overlap rate | Fraction of cells closer than D_MIN to a neighbour (real tissue ≈ 0.02) |

**Reference rows:** *Real train vs test* shows how different two real regions of the same tissue are, the best you can realistically hope for. *Random scatter* and *Type-shuffled* are the baselines the model must beat.

In [ ]:
# ---------- METRICS CELL (numpy / scipy only) ----------
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from scipy.stats import wasserstein_distance


def nn_distances(pos):
    """Distance from every cell to its nearest neighbour, pooled over patches (cell-spacing units)."""
    out = []
    for p in pos:
        d, _ = cKDTree(p).query(p, k=2)
        out.append(d[:, 1])
    return np.concatenate(out)


def pair_distances(pos, max_patches=300):
    return np.concatenate([pdist(p) for p in pos[:max_patches]])


def patch_radius(pos):
    """Radius of gyration of each patch: how spread out / compact it is."""
    return np.sqrt((pos ** 2).sum(-1).mean(-1))


def composition(typ, n_types):
    c = np.bincount(typ.ravel(), minlength=n_types).astype(float)
    return c / c.sum()


def cooccurrence(pos, typ, n_types, k=5):
    """P(neighbour type = b | cell type = a) over each cell's k nearest neighbours."""
    C = np.zeros((n_types, n_types))
    for p, t in zip(pos, typ):
        _, idx = cKDTree(p).query(p, k=k + 1)
        nb = t[idx[:, 1:]]
        np.add.at(C, (np.repeat(t, k), nb.ravel()), 1)
    rows = C.sum(1, keepdims=True)
    return C / np.maximum(rows, 1), rows.ravel()


def evaluate(gen_pos, gen_typ, ref_pos, ref_typ, n_types, d_min):
    """Compare a set of patches against reference (held-out real) patches. Lower = closer to real tissue."""
    nn_g, nn_r = nn_distances(gen_pos), nn_distances(ref_pos)
    P_g, _ = cooccurrence(gen_pos, gen_typ, n_types)
    P_r, rows_r = cooccurrence(ref_pos, ref_typ, n_types)
    w = rows_r / rows_r.sum()
    return {
        'NN-distance W1': float(wasserstein_distance(nn_g, nn_r)),
        'Pair-distance W1': float(wasserstein_distance(pair_distances(gen_pos), pair_distances(ref_pos))),
        'Patch-radius W1': float(wasserstein_distance(patch_radius(gen_pos), patch_radius(ref_pos))),
        'Composition TVD': float(0.5 * np.abs(composition(gen_typ, n_types) - composition(ref_typ, n_types)).sum()),
        'Neighbourhood TVD': float((w * 0.5 * np.abs(P_g - P_r).sum(1)).sum()),
        'Overlap rate': float((nn_g < d_min).mean()),
    }


# ---------- BASELINES ----------
def random_baseline(n, n_cells, ref_pos, type_freq, rng):
    """Cells scattered uniformly in a disc of realistic size, types drawn independently."""
    R = np.median(np.sqrt((ref_pos ** 2).sum(-1)).max(-1))
    r = R * np.sqrt(rng.random((n, n_cells)))
    a = rng.random((n, n_cells)) * 2 * np.pi
    pos = np.stack([r * np.cos(a), r * np.sin(a)], -1)
    pos -= pos.mean(1, keepdims=True)
    typ = rng.choice(len(type_freq), size=(n, n_cells), p=type_freq)
    return pos.astype(np.float32), typ


def shuffled_types_baseline(pos, typ, rng):
    """Real cell positions, but cell types shuffled within each patch: tests if arrangement of types is learned."""
    return pos.copy(), rng.permuted(typ, axis=1)

In [ ]:
n = CFG['n_eval']
ref_pos, ref_typ = test_pos[:n], test_typ[:n]
rows, samples = {}, {}

rows['Real train vs real test (floor)'] = evaluate(train_pos[:n], train_typ[:n], ref_pos, ref_typ, K, D_MIN)
rp, rt = random_baseline(n, CFG['n_cells'], train_pos, TYPE_FREQ, rng)
rows['Random scatter (baseline)'] = evaluate(rp, rt, ref_pos, ref_typ, K, D_MIN)
# Built from TRAINING patches (v1 wrongly used test patches, which gave this baseline an unfair head start)
sp, st = shuffled_types_baseline(train_pos[n:2 * n], train_typ[n:2 * n], rng)
rows['Type-shuffled real (baseline)'] = evaluate(sp, st, ref_pos, ref_typ, K, D_MIN)

LABELS = {'main': 'Flow model (main)',
          'lowdata_no_physics': f'Low-data ({ABLATION["n_train_patches"]} patches), no physics',
          'lowdata_physics': f'Low-data ({ABLATION["n_train_patches"]} patches) + overlap loss'}
for tag, label in LABELS.items():
    if tag in models:
        gp, gt = sample(models[tag], n, CFG['n_cells'], K, POS_SCALE, CFG['sample_steps'], device=DEVICE)
        samples[tag] = (gp, gt)
        rows[label] = evaluate(gp, gt, ref_pos, ref_typ, K, D_MIN)

results = pd.DataFrame(rows).T
display(results.style.format('{:.3f}').background_gradient(axis=0, cmap='RdYlGn_r'))
results.to_csv('results/metrics.csv')

# Plain-language verdict for the main model
best, rnd, flo = LABELS['main'], 'Random scatter (baseline)', 'Real train vs real test (floor)'
for m in results.columns:
    if m == 'Overlap rate':
        continue
    g, r, f = results.loc[best, m], results.loc[rnd, m], results.loc[flo, m]
    gap = f'{100 * (r - g) / (r - f):.0f}% of the way from random to real-vs-real' if r > f else 'random is already ~real on this metric'
    print(f'{m:18s}: model {g:.3f} | random {r:.3f} | real-vs-real {f:.3f} -> '
          f'{"beats random" if g < r else "DOES NOT beat random"} ({gap})')
print(f'Overlap rate      : model {results.loc[best, "Overlap rate"]:.3f} vs real ~{results.loc[flo, "Overlap rate"]:.3f}')
if RUN_ABLATION:
    a, b = LABELS['lowdata_physics'], LABELS['lowdata_no_physics']
    print(f'\nPhysics ablation (low data): overlap rate {results.loc[a, "Overlap rate"]:.3f} with penalty vs '
          f'{results.loc[b, "Overlap rate"]:.3f} without; NN-distance W1 {results.loc[a, "NN-distance W1"]:.3f} vs '
          f'{results.loc[b, "NN-distance W1"]:.3f}. Treat differences under ~0.01 as noise unless they hold across seeds.')

In [ ]:
# Real vs generated patches, side by side
tag = 'main'
gp, gt = samples[tag]
fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for i in range(5):
    for row, (P, T, name) in enumerate([(ref_pos, ref_typ, 'Real (held-out)'), (gp, gt, 'Generated')]):
        a = axes[row, i]
        a.scatter(P[i, :, 0], P[i, :, 1], c=colours(T[i]), s=90, edgecolors='k', linewidths=0.4)
        a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
        lim = 1.1 * np.abs(ref_pos[:5]).max(); a.set_xlim(-lim, lim); a.set_ylim(-lim, lim)
        if i == 0: a.set_ylabel(name, fontsize=12)
fig.suptitle('Top: real held-out tissue patches. Bottom: patches generated from pure noise.')
plt.tight_layout(); plt.savefig('results/real_vs_generated.png', dpi=150); plt.show()

# The learned flow: one sample from noise (t=0) to tissue (t=1)
_, _, traj = sample(models[tag], 1, CFG['n_cells'], K, POS_SCALE, CFG['sample_steps'], device=DEVICE, keep_traj=True)
fig, axes = plt.subplots(1, len(traj), figsize=(3.2 * len(traj), 3.4))
for a, x, tt in zip(axes, traj, np.linspace(0, 1, len(traj))):
    p = x[:, :2].numpy() * POS_SCALE
    a.scatter(p[:, 0], p[:, 1], c=colours(x[:, 2:].argmax(-1).numpy()), s=40, edgecolors='k', linewidths=0.3)
    a.set_title(f't = {tt:.2f}'); a.set_aspect('equal'); a.set_xticks([]); a.set_yticks([])
fig.suptitle('Learned flow: integrating v_theta moves random noise into a tissue patch')
plt.tight_layout(); plt.savefig('results/flow_trajectory.png', dpi=150); plt.show()

# Spacing and neighbourhood structure
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
bins = np.linspace(0, 3, 60)
ax[0].hist(nn_distances(ref_pos), bins, density=True, alpha=0.5, label='real (held-out)')
for t_, (p, _) in samples.items():
    ax[0].hist(nn_distances(p), bins, density=True, histtype='step', lw=2, label=f'generated ({t_})')
ax[0].hist(nn_distances(rp), bins, density=True, histtype='step', lw=1.5, ls='--', label='random scatter')
ax[0].axvline(D_MIN, color='k', ls=':', lw=1); ax[0].set_xlabel('nearest-neighbour distance (cell spacings)')
ax[0].legend(fontsize=8); ax[0].set_title('Cell spacing')
P_r, _ = cooccurrence(ref_pos, ref_typ, K); P_g, _ = cooccurrence(gp, gt, K)
for a, P, name in [(ax[1], P_r, 'Real: P(neighbour type | cell type)'), (ax[2], P_g, 'Generated')]:
    im = a.imshow(P, vmin=0, vmax=max(P_r.max(), P_g.max()), cmap='viridis')
    a.set_title(name); a.set_xlabel('neighbour type'); a.set_ylabel('cell type')
    a.set_xticks(range(K)); a.set_yticks(range(K)); a.set_xticklabels(range(K), fontsize=6); a.set_yticklabels(range(K), fontsize=6)
plt.colorbar(im, ax=ax[2], fraction=0.046)
plt.tight_layout(); plt.savefig('results/spacing_and_neighbourhoods.png', dpi=150); plt.show()
print('Type index -> name:', dict(enumerate(TYPE_NAMES)))

In [ ]:
summary = {'config': CFG, 'dataset_type_column': type_key, 'n_cells_total': int(len(coords)), 'n_types': K,
           'd_min': D_MIN, 'pos_scale': POS_SCALE, 'metrics': results.to_dict(orient='index')}
json.dump(summary, open('results/summary.json', 'w'), indent=2, default=float)
!zip -qr morphix_stage1_results.zip results
print('Saved results/ and morphix_stage1_results.zip (figures, metrics.csv, summary.json, checkpoints)')
try:
    from google.colab import files; files.download('morphix_stage1_results.zip')
except Exception:
    pass

## 5. How to read the results, and what's honest to claim

**A good Stage 1 result:** the main model beats random scatter and type-shuffled tissue, especially on **Neighbourhood TVD** (are the right cell types next to each other?), and gets a large share of the way to the *real vs real* row. The verdict lines print that share for you.

**Metrics that can't discriminate:** on some datasets random scatter is already close to real on a metric (v1: NN-distance W1 and overlap rate). A model can't meaningfully 'beat' that, so don't present it as a win or a failure; say the metric isn't informative for this tissue.

**Physics ablation:** only claim the overlap loss helps if the low-data model with the penalty has a clearly lower overlap rate (and no worse other metrics) **across 3 seeds**. If it doesn't, the honest finding is: *"the model learns physically plausible spacing from data alone; an explicit overlap prior gave no measurable benefit"*. That's a perfectly good result to report.

**What you can claim if it works:** *"A permutation-equivariant flow-matching model trained on spatial single-cell data generates tissue neighbourhoods whose geometry and cell-type neighbourhood structure match held-out tissue, closing X% of the gap between random and real-vs-real on neighbourhood structure."* Put in your real numbers.

**Limitations (state these yourself before a reviewer does):**
- 2D patches of 64 cells from one tissue; not 3D, not whole tissues.
- No genomic or perturbation conditioning yet.
- Cell types are generated with a continuous relaxation plus argmax, which is simple but not the most principled discrete method.
- A single seed; for a write-up, change `seed` to 0, 1, 2 and report mean ± std.
- Generated cell types can look 'salt-and-pepper' (too many types mixed in one patch); the separate held-out type loss shows how much this improves with training.

**Next (Stage 2):** condition the velocity field on a perturbation or region label, $v_\theta(x_t, t \mid z)$, train on paired perturbation–morphology data, and test on perturbations held out from training.